# smock for large galaxies: the walkthrough with the stars in chunks

The walkthrough holds every star's spectra in memory at once: about 0.6 MB per star with its emission model, so a galaxy of a few hundred thousand stars doesn't fit even on a 240 GB node, and those are often the most interesting ones. This notebook runs the same steps with the stars split into chunks, so the memory depends on the chunk size rather than on the galaxy. Its results are the walkthrough's, to rounding (about 1 part in 10^14).

**Read the walkthrough first:** the notes here only cover what changes. Its code cells are reused here unchanged, apart from these steps:

- **6. The instrument** comes before the spectra (the chunks make their images as they go).
- **7. How many stars at a time.** A small trial chunk measures what your emission model costs per star, and the chunk size takes a cautious share of the memory left.
- **8. Spectra, photometry and images, chunk by chunk (pass 1).** Each chunk's stars go through the stellar part of the model. Their summed spectra, photometry and images are kept; their per-star spectra are dropped before the next chunk. The black holes go through the AGN part once. The galaxy's spectra and images are the sums of the parts, as in the model.
- **12. The spectrum inside the source.** The walkthrough adds up the per-star spectra of the stars inside the source, which no longer exist here. By default the chunks run a second time (pass 2) and add up only those stars: exactly the walkthrough's answer, for about twice the computing time. With `APERTURE_SPECTRUM = "preset"` the spectrum is instead taken inside a circle fixed in advance (`PRESET_APERTURE_ARCSEC` about the catalogue position) and added up during pass 1: one pass, but not the source's own aperture.

1. Paths and the galaxy
2. Load the particles
3. Pass that particle data to synthesizer
4. Line-of-sight dust
5. The emission model
6. The instrument
7. How many stars at a time
8. Spectra, photometry and images, chunk by chunk (pass 1)
9. Saving the spectrum and photometry
10. Image segmentation
11. Galaxy Morphology (optional)
12. Re-measuring the galaxy's properties within the segmented area (pass 2)
13. Saving the spectrum and the image data
14. Creating Figures
15. Is it a little red dot? (optional)

On a cluster, run this on a compute node with the memory you want it to use: the chunk size follows the job's allocation.

In [ ]:
import warnings

import numpy as np
import matplotlib.pyplot as plt
from unyt import K, Myr, angstrom, arcsec, km, nJy, s, unyt_array

from synthesizer.emission_models import (AttenuatedEmission, BlackHoleEmissionModel, DustEmission,
                                         EmissionModel, LOSStellarEmission)
from synthesizer.emission_models.attenuation import Inoue14
from synthesizer.emission_models.generators.dust import Greybody
from synthesizer.emission_models.transformers import CoveringFraction, EscapingFraction, PowerLaw
from synthesizer.emissions import plot_observed_spectra, plot_spectra
from synthesizer.grid import Grid
from synthesizer.instruments import FilterCollection, PhotometricImager
from synthesizer.kernel_functions import Kernel
from synthesizer.particle import BlackHoles, Galaxy, Gas, Stars

from smock import agn, config, imaging
from smock.data import code_mass_to_msun, crop, kappa_code_to_physical, load_galaxy, open_snapshot, subhalo_row
from smock.dust import WindowedPowerLaw

In [ ]:
from synthesizer.imaging import ImageCollection
from synthesizer.pipeline.pipeline_utils import accumulate_pipeline_results_from_child

from smock import memory

## 1. Paths and the galaxy

As in the walkthrough, but by default the test cutout's largest group, 42595 (54k stars, about 33 GB if done all at once), and its own output folder.

In [ ]:
# Simulation input. On Vera the full ASTRID snapshots are at config.EXTERNAL["astrid_pig_dir"].
PIG_DIR = config.TEST_CUTOUT_PIG_DIR
SNAP = 47                       # PIG_047_subfind: z = 6

# Synthesizer data: the old pipeline's stellar grid (BPASS 2.2.1, binaries, Chabrier IMF, CLOUDY c23.01)
GRID_DIR = config.GRID_DIR / "sps/photoionised/bpass"
GRID_NAME = "bpass-2.2.1-bin_chabrier03-0.1,300.0_cloudy-c23.01-sps"

# The AGN grid: CLOUDY model using CLOUDY's "agn" continuum, with alpha_ox = -1.4 and alpha_uv = -0.5 (made by smock-recipes/agn_grid).
# standard_blr_core: BLR-like clouds (the default; their parameters are set in step 3)
# gas_enshrouded_core: The AGN is enshrouded in dense gas. The model used by the old pipeline when searching for LRDs
INCLUDE_AGN = True              # False: stars only
AGN_GRID_DIR = config.GRID_DIR / "agn/photoionised/agn-cloudy"
AGN_GRID_NAME = "agn-cloudy_T20_aox-1.40_auv-0.50_cloudy-standard_blr_core"

# Produced data
OUT_DIR = config.OUTPUT_DIR / "walkthrough_large_galaxy"

# The galaxy, by its subhalo index in the full ASTRID snapshot. The test set:
#   303604 typical star-forming galaxy with an accreting BH   54437 LRD-like (compact, bright BH)
#   24900 two galaxies 1.8" apart   4283857 small (102 stars)   42595 massive group (54k stars)
SIDX = 42595
PARTICLES = "group"             # load every particle of its FoF group, so neighbours in the field show too;
                                # or "subhalo": that subhalo alone (a satellite then looks isolated)

# The images
PIXEL = 0.03                    # arcsec
FOV = 3.0                       # arcsec: the saved images are 100 x 100 pixels
BUFFER_PIX = 5                  # extra pixels imaged per side, so the saved images can be re-centred by cropping
IMAGE_FOV = FOV + 2 * BUFFER_PIX * PIXEL

print(f"snapshots: {PIG_DIR}\ngrid:      {GRID_DIR / GRID_NAME}.hdf5\n"
      f"AGN grid:  {AGN_GRID_DIR / AGN_GRID_NAME}.hdf5\noutputs:   {OUT_DIR}")

## 2. Load the particles

As in the walkthrough.

In [ ]:
snap = open_snapshot(PIG_DIR, SNAP)
row = subhalo_row(snap, SIDX)
gal = load_galaxy(snap, sidx=row, particles=PARTICLES)
# the subhalo's stellar mass in the SUBFIND catalogue (current masses), for step 15's checks
m_star_subfind = float(code_mass_to_msun(snap.file["SubGroups/SubhaloMassType"][row:row + 1][0, 4], snap.h))

CROP_SHAPE = "prism"            # square in x-y, any z; or "cylinder" (the old pipeline); or "cube"
CROP_UNIT = "arcsec"            # or "kpc", "ckpc/h", "r_half" (stellar half-mass radii)
KERNEL_MARGIN_PIX = 10
CROP_SIZE = IMAGE_FOV / 2 + KERNEL_MARGIN_PIX * PIXEL    # half-width in CROP_UNIT (arcsec here)
GAS_MARGIN_CKPCH = 5.0          # comoving kpc/h, as in the old pipeline
gal = crop(gal, CROP_SIZE, CROP_UNIT, shape=CROP_SHAPE, gas_margin_ckpch=GAS_MARGIN_CKPCH)

z, cosmo = snap.redshift, snap.cosmo
print(f"z = {z:.3f}: {len(gal.stars['ages'])} stars, {len(gal.gas['masses'])} gas particles, "
      f"{len(gal.bhs['ids'])} black holes")
print(f"crop: {CROP_SHAPE}, half-width {CROP_SIZE:.2f} {CROP_UNIT} = {gal.crop['size_kpc']:.2f} kpc")
print(f"stellar mass: {gal.stars['initial_masses'].sum():.3e} at birth, "
      f"{gal.stars['current_masses'].sum():.3e} now (SUBFIND: {m_star_subfind:.3e} for the subhalo)")

## 3. Pass that particle data to synthesizer

As in the walkthrough.

In [ ]:
STAR_MASS = "initial_masses"    # birth mass; "current_masses" reproduces the old pipeline
DUST_TO_METAL = 1.0             # the opacity in step 4 is per unit metal mass, so dust = metals

S, G = gal.stars, gal.gas
stars = Stars(
    initial_masses=S[STAR_MASS],
    current_masses=S["current_masses"],
    ages=S["ages"],
    metallicities=S["metallicities"],
    coordinates=S["coordinates"],
    smoothing_lengths=S["smoothing_lengths"],
    redshift=z,
)
gas = Gas(
    masses=G["masses"],
    metallicities=G["metallicities"],
    coordinates=G["coordinates"],
    smoothing_lengths=G["smoothing_lengths"],   # MP-Gadget's SmoothingLength is the kernel support radius, as synthesizer expects
    dust_to_metal_ratio=DUST_TO_METAL,
    star_formation_rates=G["star_formation_rates"],   # Msun/yr; synthesizer keeps extra arrays as attributes (step 12's SFR map)
    redshift=z,
)

# ---- black holes ----
EFFICIENCY = 0.1                # radiative efficiency: L_bol = EFFICIENCY * Mdot c^2 (ASTRID, AMBRA)
DISC_TEMPERATURE = "T_max"      # T_max at (49/36) r_isco (the old pipeline); "radius" (DISC_RADIUS); or a number, in K
DISC_RADIUS = 3.0               # with DISC_TEMPERATURE = "radius"
DISC_RADIUS_UNIT = "r_isco"     # "r_isco", "r_g" (GM/c^2) or "R_s"
R_ISCO = 6.0                    # GM/c^2: non-spinning
# One point on the grid's other axes for every BH. The standard BLR's fiducial values:
AGN_GRID_POINT = dict(ionisation_parameter=10**-1.5, hydrogen_density=1e10, column_density=23.0)
# the gas-enshrouded grid (the old pipeline's values) also has turbulence and metallicity axes:
# AGN_GRID_POINT = dict(ionisation_parameter=10**-1.5, hydrogen_density=10**9.5, column_density=24.5,
#                       turbulence=400.0 * km / s, metallicities=0.0015)
# (hydrogen density, n_H, in cm^-3; column_density, N_H, is log10 N_H / cm^-2; metallicity is Z, here 0.1 Z_sun)
# Testing only: AGN_U_FROM = "old_pipeline" gives each BH the U its disc temperature had in the old
# pipeline's CLOUDY runs, which fixed L = 10^43.5 erg/s and r = 10^16.7 cm at n_H = 10^9.5, so U rose
# with T (log U -2.8 at log T 4.2, -1.5 at 5.0, -1.0 at 6.0).
AGN_U_FROM = "grid_point"       # or "old_pipeline"

# ---- NOTE: the covering fraction, and a slightly unusual default ----
# f = 1 means every line of sight to the disc passes through a cloud: we never see the bare
# disc, only light transmitted through the clouds, plus the clouds' own emission. That is what
# the old pipeline did and suits the gas-enshrouded picture, but BLR covering
# fractions are usually put at ~0.1-0.5 at low redshift. For f < 1, COVERING_MODE sets how a BH's line of sight is
# treated:
#   "random" (default): it has a probability f to pass through a BLR cloud, a yes/no draw per BH. With
#       COVERING_SEED = "bh_id_and_snapshot" each draw is reproducible but changes between
#       snapshots (the clouds move); COVERING_SEED = None draws truly randomly each run.
#   "weighted_combination": every BH shows the average, (1 - f) bare disc + f transmitted.
# The clouds' own emission (the grid's "nebular") is scaled by f in both modes.
COVERING_FRACTION = 1.0
COVERING_MODE = "random"
COVERING_SEED = "bh_id_and_snapshot"

B = gal.bhs
n_bh = len(B["ids"])
agn_on = INCLUDE_AGN and n_bh > 0
black_holes = None
if n_bh > 0:
    # every BH, AGN or not, so steps 10-12 can count them. A BH with Mdot = 0 (a few in AMBRA,
    # none in ASTRID) emits nothing; its disc temperature is 0 K, so synthesizer warns about
    # log10(0) and the grid check below flags it: a sign of an unusual BH, not an error
    mass, mdot, ids = B["masses"], B["accretion_rates"], B["ids"]
    if DISC_TEMPERATURE == "T_max":
        t_disc = agn.disc_max_temperature(mass, mdot, r_isco=R_ISCO)
    elif DISC_TEMPERATURE == "radius":
        t_disc = agn.disc_temperature_at(mass, mdot, DISC_RADIUS, unit=DISC_RADIUS_UNIT, r_isco=R_ISCO)
    else:
        t_disc = np.full(n_bh, float(DISC_TEMPERATURE)) * K
    bh_grid_point = dict(AGN_GRID_POINT)
    if AGN_U_FROM == "old_pipeline":
        incident_grid = config.GRID_DIR / "agn/incident/agn-cloudy" / (AGN_GRID_NAME.split("_cloudy-")[0] + ".hdf5")
        bh_grid_point["ionisation_parameter"] = agn.old_pipeline_ionisation_parameter(
            agn.ionising_photons_per_lbol(incident_grid, t_disc))
    if COVERING_MODE == "weighted_combination":
        los_covered = np.full(n_bh, COVERING_FRACTION)
    else:
        seeds = [None if COVERING_SEED is None else [int(i), SNAP] for i in ids]
        los_covered = np.array([float(np.random.default_rng(sd).random() < COVERING_FRACTION) for sd in seeds])
    black_holes = BlackHoles(
        masses=mass,
        accretion_rates=mdot,
        epsilons=EFFICIENCY,                        # synthesizer: L_bol = epsilon * Mdot c^2
        coordinates=B["coordinates"],
        ids=ids,
        # synthesizer 1.3.1 images BHs as points but checks for smoothing lengths first, so we give them all a 1 pc smoothing length to pass that check
        smoothing_lengths=unyt_array(np.full(n_bh, 1e-3), "kpc"),
        big_bump_temperature=t_disc.to("K").value,  # log axes take plain values in the axis's units
        **{name: value * np.ones(n_bh) for name, value in bh_grid_point.items()},
        agn_covering_fraction=np.full(n_bh, COVERING_FRACTION),   # f: scales the clouds' own emission
        agn_los_covered=los_covered,                # f, or the 0/1 draw: transmitted vs bare disc
        redshift=z,
    )
galaxy = Galaxy(stars=stars, gas=gas, black_holes=black_holes, redshift=z,
                centre=unyt_array([0.0, 0.0, 0.0], "kpc"))

print(f"{n_bh} black holes, {int((B['accretion_rates'].value > 0).sum())} accreting" + ("" if agn_on else "; no AGN"))
if black_holes is not None:
    for i in range(n_bh):
        print(f"  BH {ids[i]}: log M = {np.log10(mass[i].value):.2f}, Mdot = {mdot[i].value:.3g} Msun/yr, "
              f"log L_bol = {np.log10(black_holes.bolometric_luminosities[i].to('erg/s').value):.2f}, "
              f"log T = {np.log10(t_disc[i].value):.2f}, line of sight covered: {los_covered[i]:g}")

# BHs off the grid are clamped to its edge by synthesizer without a warning; and some grid
# points are estimates (CLOUDY limit cycles, or filled-in failures; see the grid report in smock-recipes/agn_grid)
if agn_on:
    report = agn.grid_check(AGN_GRID_DIR / f"{AGN_GRID_NAME}.hdf5", big_bump_temperature=t_disc,
                            **{k: v * np.ones(n_bh) for k, v in bh_grid_point.items()})
    for i, r in enumerate(report):
        if r["outside"] or r["flags"]:
            print(f"BH {ids[i]}: outside the grid in {r['outside'] or '-'}; uses grid points flagged {r['flags'] or '-'}")
    print(f"AGN grid check: {sum(bool(r['outside'] or r['flags']) for r in report)} of {n_bh} BHs flagged")


## 4. Line-of-sight dust

As in the walkthrough. The optical depths are computed for every star at once (they need all the gas), and each chunk carries its own stars' values.

In [ ]:
KAPPA_CODE = 10**4.1            # ISM dust opacity in the old pipeline's calibration units
KAPPA_BC = 1.0                  # birth-cloud tau_V at Z = Z_BC, as in FLARES; the old pipeline used 3
Z_BC = 0.01
kappa = kappa_code_to_physical(KAPPA_CODE, snap.a, snap.h)   # pc^2 / Msun of metals
SPH_KERNEL = "quintic"          # MP-Gadget's kernel in ASTRID and AMBRA (tests/test_kernel.py)
kernel = Kernel(SPH_KERNEL)

galaxy.get_stellar_los_tau_v(kappa=kappa, kernel=kernel)    # stored as galaxy.stars.tau_v
tau = galaxy.stars.tau_v
print(f"kappa = {kappa:.4f} pc^2/Msun; tau_V median {np.median(tau):.3f}, "
      f"95th percentile {np.percentile(tau, 95):.3f}, max {tau.max():.2f}")

if agn_on:
    galaxy.get_black_hole_los_tau_v(kappa=kappa, kernel=kernel)   # BHs as points: galaxy.black_holes.tau_v
    print(f"black holes: tau_V = {np.round(galaxy.black_holes.tau_v, 3)}")

# birth clouds: stars younger than YOUNG_AGE (step 5) also sit behind their cloud's dust
galaxy.stars.young_tau_v = KAPPA_BC * galaxy.stars.metallicities / Z_BC


## 5. The emission model

As in the walkthrough. Its stellar part (`stellar_dust`, which holds `stellar_total`) runs on the chunks, and its AGN part (`agn_dust`) on the black holes.

In [ ]:
grid = Grid(GRID_NAME, grid_dir=str(GRID_DIR))

YOUNG_AGE = 10 * Myr             # younger stars sit in their birth clouds (FLARES)
DUST_SLOPE_BC = -1.0            # power-law dust curves: birth clouds, ISM, AGN (FLARES)
DUST_SLOPE_ISM = -1.0
AGN_DUST_SLOPE = DUST_SLOPE_ISM  # the AGN sits behind the same ISM dust (the old pipeline used -2)
AGN_DUST_WINDOW = (20 * angstrom, 1e6 * angstrom)   # the AGN's dust acts at 2 nm - 100 um only; None: everywhere
DUST_TEMPERATURE = 100 * K      # greybody dust emission, for the stars and the AGN
DUST_BETA = 2.0


# ---- stars ----
stellar = LOSStellarEmission(grid, young_upper_limit=YOUNG_AGE, birth_cloud_slope=DUST_SLOPE_BC,
                             ism_slope=DUST_SLOPE_ISM)        # its top model is "stellar_total"
stellar_dust = DustEmission(
    dust_emission_model=Greybody(DUST_TEMPERATURE, emissivity=DUST_BETA),   # one Greybody per DustEmission
    dust_lum_intrinsic=stellar["reprocessed"],    # energy balance: absorbed = intrinsic - attenuated
    dust_lum_attenuated=stellar,
    emitter="stellar",
    label="stellar_dust_emission",
)
per_particle = [stellar, stellar_dust]
totals, dust_parts, reprocessed_parts = [stellar], [stellar_dust], [stellar["reprocessed"]]

# ---- AGN ----
if agn_on:
    agn_grid = Grid(AGN_GRID_NAME, grid_dir=str(AGN_GRID_DIR), new_lam=grid.lam)   # spectra add on one wavelength array
    los, f_attr = ("agn_los_covered",), ("agn_covering_fraction",)
    disc = BlackHoleEmissionModel("AGN_incident", grid=agn_grid, extract="incident")
    through_clouds = BlackHoleEmissionModel("AGN_transmitted_full", grid=agn_grid, extract="transmitted")
    clouds = BlackHoleEmissionModel("AGN_nebular_full", grid=agn_grid, extract="nebular")
    agn_reprocessed = BlackHoleEmissionModel("AGN_reprocessed", combine=(
        BlackHoleEmissionModel("AGN_escaped", apply_to=disc, transformer=EscapingFraction(covering_attrs=los)),
        BlackHoleEmissionModel("AGN_transmitted", apply_to=through_clouds, transformer=CoveringFraction(covering_attrs=los)),
        BlackHoleEmissionModel("AGN_nebular", apply_to=clouds, transformer=CoveringFraction(covering_attrs=f_attr)),
    ))
    agn_dust_curve = (PowerLaw(slope=AGN_DUST_SLOPE) if AGN_DUST_WINDOW is None
                      else WindowedPowerLaw(AGN_DUST_SLOPE, *AGN_DUST_WINDOW))
    agn_total = AttenuatedEmission(dust_curve=agn_dust_curve, apply_to=agn_reprocessed,
                                   emitter="blackhole", tau_v="tau_v", label="AGN_total")
    agn_dust = DustEmission(
        dust_emission_model=Greybody(DUST_TEMPERATURE, emissivity=DUST_BETA),
        dust_lum_intrinsic=agn_reprocessed,
        dust_lum_attenuated=agn_total,
        emitter="blackhole",
        label="AGN_dust_emission",
    )
    per_particle += [agn_total, agn_dust]
    totals.append(agn_total)
    dust_parts.append(agn_dust)
    reprocessed_parts.append(agn_reprocessed)

# ---- the galaxy ----
total_dust = EmissionModel("total_dust_emission", combine=dust_parts, emitter="galaxy")
reprocessed_total = EmissionModel("reprocessed_total", combine=reprocessed_parts, emitter="galaxy")
model = EmissionModel("total", combine=(*totals, total_dust), emitter="galaxy",
                      related_models=(reprocessed_total,))

for m in per_particle:
    m.set_per_particle(True)
model.set_save(False, set_all=True)
SAVED = ["total", "reprocessed_total", "total_dust_emission", "stellar_total", "reprocessed",
         "stellar_dust_emission"] + (["AGN_total", "AGN_reprocessed", "AGN_dust_emission"] if agn_on else [])
for label in SAVED:
    model[label].set_save(True)

fig, ax = model.plot_emission_graph(show=False, figsize=(16, 8))
plt.show()

## 6. The instrument

As in the walkthrough's step 7, with the filters and the IGM model from its step 6.

In [ ]:
IGM = Inoue14                   # IGM absorption model (Inoue et al. 2014), for the stars and the AGN
FILTER_CODES = [f"JWST/NIRCam.{f}" for f in imaging.NIRCAM_WIDE]
filters = FilterCollection(filter_codes=FILTER_CODES)   # observed-frame curves, applied to the observed spectrum

In [ ]:
# PIXEL and FOV are set in step 1
PSF_SOURCE = "stpsf"            # or "gaussian"
PSF_PRODUCT = "DET_SAMP"        # STPSF's detector-sampled PSF; "OVERSAMP" is the 0.006" one
NOISE_PRESET = "JADES-Medium"   # or "CEERS"
R_AP = imaging.SURVEY_APERTURE_RADIUS_ARCSEC

psfs, depths = {}, {}
for fc in FILTER_CODES:
    f = fc.split(".")[-1]
    if PSF_SOURCE == "stpsf":
        psfs[fc], header = imaging.load_stpsf_psf(config.PSF_DIR, f, PSF_PRODUCT)
        ee = header["EE_R0P1"]          # measured on STPSF's 0.006" grid
    else:
        psfs[fc] = imaging.gaussian_psf(imaging.NIRCAM_FWHM_ARCSEC[f], PIXEL)
        ee = imaging.encircled_energy(psfs[fc], PIXEL, R_AP)
    depth_njy = imaging.ab_mag_to_njy(imaging.SURVEY_DEPTHS_5SIGMA[NOISE_PRESET][f])
    sigma_pix = imaging.sigma_pix_from_depth(depth_njy, 5, R_AP, PIXEL, ee)
    depths[fc] = 5 * sigma_pix * nJy            # per-pixel 5-sigma depth: synthesizer draws sigma = depth / 5
    print(f"{f}: depth {imaging.SURVEY_DEPTHS_5SIGMA[NOISE_PRESET][f]:.2f} AB = {depth_njy:5.2f} nJy, "
          f"EE(0.1\") = {ee:.2f}, sigma_pix = {sigma_pix:.3f} nJy")

instrument = PhotometricImager(
    label="NIRCam_0.03",
    filters=filters,
    resolution=PIXEL * arcsec,
    psfs=psfs,
    depth=depths,
    snrs=5.0,
)

## 7. How many stars at a time

- `CHUNK_STARS = "auto"` runs a small trial chunk (`TRIAL_STARS`) through everything a chunk does and measures how far the memory rises. That cost grows in proportion to the number of stars, so it gives the cost per star for *your* emission model: saving more spectra per star, or a grid with more wavelengths, costs more. (Off Linux, where this can't be measured, it estimates the cost from the number of saved spectra instead.)
- The chunk size is `MEMORY_SAFETY` × the memory left ÷ the cost per star. The memory left is the job's allocation (from SLURM, or `MEMORY_LIMIT_GB`) minus what this notebook already uses, mostly the grids.
- `MEMORY_SAFETY = 0.5` is deliberately cautious. A few extra chunks only cost a little time, while a chunk slightly too big ends the run with an out-of-memory error.
- `CHUNK_STARS` can also be a number of stars, or `None` for all of them at once (as the walkthrough does).

`APERTURE_SPECTRUM` chooses how step 12 gets the spectrum inside the source (see the introduction). For `"preset"`, the circle is `PRESET_APERTURE_ARCSEC` in radius about the catalogue position, the centre of the images.

In [ ]:
CHUNK_STARS = "auto"            # stars per chunk: "auto", a number, or None (all at once)
MEMORY_LIMIT_GB = None          # None: this job's allocation (SLURM), else the machine's memory; or a number
MEMORY_SAFETY = 0.5             # use at most this share of the memory left (cautious on purpose)
TRIAL_STARS = 500               # stars in the trial chunk that measures the cost per star
APERTURE_SPECTRUM = "two_pass"  # or "preset": a fixed circle, added up during pass 1 (one pass)
PRESET_APERTURE_ARCSEC = 0.5    # its radius, about the catalogue position

STAR_LABELS = [label for label in SAVED if model[label].emitter == "stellar"]
BH_LABELS = [label for label in SAVED if model[label].emitter == "blackhole"]
STAR_IMAGE_LABELS = ["stellar_total", "stellar_dust_emission"]   # the stars' parts of the total image
BH_IMAGE_LABELS = ["AGN_total", "AGN_dust_emission"] if agn_on else []

# a copy of the instrument with its PSFs but no noise: chunk images are added up, and the noise
# goes on the sum once (step 8)
imager = PhotometricImager(label=instrument.label, filters=filters, resolution=instrument.resolution, psfs=psfs)
image_kw = dict(fov=IMAGE_FOV * arcsec, instrument=imager, kernel=kernel.get_kernel(), cosmo=cosmo)


def run_chunk(chunk, weights=None):
    """One chunk of stars through the stellar part of the model: spectra, observed spectra,
    photometry, per-star photometry and images, kept on the chunk. With weights (one per star
    in the chunk), also returns the weighted sums of its saved per-star spectra [erg/s/Hz]."""
    with np.errstate(over="ignore"):         # the greybody's harmless overflow, as in the walkthrough
        chunk.get_spectra(stellar_dust)
    chunk.get_observed_spectra(cosmo=cosmo, igm=IGM)
    chunk.get_photo_fnu(filters)
    chunk.stars.get_particle_photo_fnu(filters)
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message=r"\s*Coordinates must be centered")
        chunk.get_images_flux(*STAR_IMAGE_LABELS, **image_kw)
    if weights is None:
        return None
    return {p: weights @ chunk.stars.particle_spectra[p].lnu.to("erg/s/Hz").value for p in STAR_LABELS}


# the stars alone, as a galaxy synthesizer can split (the same Stars object, not a copy)
stars_only = Galaxy(stars=galaxy.stars, redshift=z, centre=galaxy.centre)
n_star = galaxy.stars.nparticles
free = memory.free_gb(MEMORY_LIMIT_GB)
if CHUNK_STARS == "auto" and free is None:
    raise RuntimeError("can't tell how much memory is free here: set MEMORY_LIMIT_GB, or CHUNK_STARS to a number")
if CHUNK_STARS == "auto" and n_star <= TRIAL_STARS:
    chunk_stars = n_star                    # a small galaxy: all at once
elif CHUNK_STARS == "auto":
    trial = stars_only.split(TRIAL_STARS)[0]
    n_trial = trial.stars.nparticles
    _, rise = memory.peak_rise_gb(run_chunk, trial, np.ones(n_trial))
    if rise is None:
        rise = memory.spectra_per_star_estimate(stellar_dust) * len(grid.lam) * 8 / 1e9 * n_trial
    gb_per_star = rise / n_trial
    del trial
    chunk_stars = min(n_star, memory.stars_per_chunk(free, gb_per_star, MEMORY_SAFETY))
    print(f"trial chunk: {n_trial} stars raised the memory by {rise:.2f} GB: {gb_per_star * 1e3:.2f} MB per star; "
          f"all {n_star} stars at once would need about {gb_per_star * n_star:.1f} GB")
else:
    chunk_stars = n_star if CHUNK_STARS is None else int(CHUNK_STARS)
n_chunks = -(-n_star // chunk_stars)
gb = lambda x: "unknown" if x is None else f"{x:.1f} GB"
print(f"memory left: {gb(free)} (now using {gb(memory.rss_gb())}) -> {n_chunks} chunk(s) of up to {chunk_stars} stars")

kpc_per_arcsec = cosmo.kpc_proper_per_arcmin(z).to("kpc/arcsec").value   # physical kpc
star_xy = galaxy.stars.centered_coordinates[:, :2].to("kpc").value
in_preset = np.hypot(*star_xy.T) <= PRESET_APERTURE_ARCSEC * kpc_per_arcsec
if APERTURE_SPECTRUM == "preset":
    print(f"preset aperture: r = {PRESET_APERTURE_ARCSEC}\" = {PRESET_APERTURE_ARCSEC * kpc_per_arcsec:.2f} pkpc, "
          f"{in_preset.sum()} stars")

## 8. Spectra, photometry and images, chunk by chunk (pass 1)

synthesizer's `Galaxy.split` cuts the stars into chunks (each chunk shares the galaxy's star arrays rather than copying them, optical depths included), and `accumulate_pipeline_results_from_child` adds each chunk's spectra, photometry and images onto the stars, as synthesizer's own `Pipeline` does when it chunks. Each chunk's per-star spectra are dropped before the next one. (`Galaxy.split` puts the black holes on the first chunk only, which the full model can't run on, hence stars and black holes separately.)

The black holes then go through the AGN part of the model, all at once (there are few). The galaxy-level spectra are the sums of their parts, as the model defines them: `total` = stars + AGN + both dust emissions. The total image is the sum of the parts' images, and its noise is drawn once, as in the walkthrough's step 8. The variables left at the end (`owner`, `raw`, `psf`, `noisy`, `alone`) are the walkthrough's, so steps 9 to 15 run unchanged.

In [ ]:
import time

SEED = 1                        # as in the walkthrough: with SNAP and SIDX, each galaxy its own noise
IMAGE_LABELS = ["total", "stellar_total"] + (["AGN_total"] if agn_on else [])

# ---- the stars, chunk by chunk ----
preset_lnu = {p: 0.0 for p in STAR_LABELS}     # stellar part of the preset-aperture spectrum
# split gives a list of every chunk; each is taken off the list as it runs, so once it's done
# nothing holds on to it and Python frees its per-star spectra (as synthesizer's Pipeline does)
chunks = stars_only.split(chunk_stars)
t0, start, peaks = time.time(), 0, []
for k in range(n_chunks):
    chunk = chunks.pop(0)
    n_c = chunk.stars.nparticles
    weights = in_preset[start:start + n_c].astype(float) if APERTURE_SPECTRUM == "preset" else None
    # the peak while the chunk runs: the memory before it, plus how far it rose (None off Linux)
    before = memory.rss_gb()
    sums, rise = memory.peak_rise_gb(run_chunk, chunk, weights)
    peaks.append(None if rise is None else before + rise)
    if sums is not None:
        for p in STAR_LABELS:
            preset_lnu[p] = preset_lnu[p] + sums[p]
    if chunk is not stars_only:             # one chunk: split returns the galaxy itself, results already on it
        accumulate_pipeline_results_from_child(stars_only, chunk)   # onto galaxy.stars
    start += n_c
    del chunk, sums
    print(f"chunk {k + 1}/{n_chunks}: {n_c} stars, {time.time() - t0:.0f} s, peak {gb(peaks[-1])} while it ran, "
          f"{gb(memory.rss_gb())} once freed")

# ---- the black holes, all at once ----
if agn_on:
    bhs_only = Galaxy(black_holes=galaxy.black_holes, redshift=z, centre=galaxy.centre)
    with np.errstate(over="ignore"):
        bhs_only.get_spectra(agn_dust)
    bhs_only.get_observed_spectra(cosmo=cosmo, igm=IGM)
    bhs_only.get_photo_fnu(filters)
    galaxy.black_holes.get_particle_photo_fnu(filters)
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message=r"\s*Coordinates must be centered")
        bhs_only.get_images_flux(*BH_IMAGE_LABELS, **image_kw)

# ---- the galaxy: each galaxy-level spectrum is the sum of its parts ----
owner = {label: galaxy if model[label].emitter == "galaxy" else
         galaxy.stars if model[label].emitter == "stellar" else galaxy.black_holes for label in SAVED}
parts = {label: [label] for label in SAVED if model[label].emitter != "galaxy"}
parts["total_dust_emission"] = [p for p in ("stellar_dust_emission", "AGN_dust_emission") if p in parts]
parts["reprocessed_total"] = [p for p in ("reprocessed", "AGN_reprocessed") if p in parts]
parts["total"] = [p for p in ("stellar_total", "AGN_total") if p in parts] + parts["total_dust_emission"]
for label in SAVED:
    if model[label].emitter == "galaxy":
        galaxy.spectra[label] = sum((owner[p].spectra[p] for p in parts[label][1:]), owner[parts[label][0]].spectra[parts[label][0]])
        galaxy.spectra[label].get_fnu(cosmo=cosmo, z=z, igm=IGM)
        galaxy.photo_fnu[label] = galaxy.spectra[label].get_photo_fnu(filters)

# ---- the images: the total is the sum of the parts' images; noise once, on the sum ----
inst = instrument.label
raw = sum((owner[p].images_fnu[inst][p] for p in parts["total"][1:]), owner["stellar_total"].images_fnu[inst]["stellar_total"])
psf = sum((owner[p].images_psf_fnu[inst][p] for p in parts["total"][1:]), owner["stellar_total"].images_psf_fnu[inst]["stellar_total"])
np.random.seed([SEED, SNAP, SIDX])
noisy = ImageCollection(resolution=psf.resolution, fov=psf.fov,
                        imgs={fc: instrument.apply_noise(psf[fc], fc, aperture_radius=instrument.depth_app_radius)
                              for fc in FILTER_CODES})
alone = {label: (owner[label].images_fnu[inst][label], owner[label].images_psf_fnu[inst][label])
         for label in IMAGE_LABELS[1:]}

print(f"pass 1 done in {time.time() - t0:.0f} s; highest peak {gb(None if None in peaks else max(peaks))}")
for fc in FILTER_CODES:
    print(f"{fc.split('.')[-1]}: photometry {galaxy.photo_fnu['total'][fc].to('nJy').value:7.2f} nJy, "
          f"image sum {raw[fc].arr.sum():7.2f} (raw), {psf[fc].arr.sum():7.2f} (PSF)")

In [ ]:
SHOWN = ["reprocessed_total", "stellar_total", "total_dust_emission", "total"] + (["AGN_total"] if agn_on else [])
spectra = {label: owner[label].spectra[label] for label in SHOWN}
fig, ax = plot_spectra(spectra, figsize=(8, 4.5), xlimits=(500, 2e6), ylimits=(1e24, 1e31))
ax.set_title("rest frame")
fig, ax = plot_observed_spectra(spectra, redshift=z, figsize=(8, 4.5),
                                xlimits=(5e3, 6e4), ylimits=(1e-2, 1e3))
ax.set_title(f"observed, z = {z:.2f}")

# Filter curves and photometry at their observed wavelengths. (We don't use
# plot_observed_spectra(filters=...): synthesizer 1.3.1 draws them shifted by another 1 + z.
# The photometry itself is right.)
tax = ax.twinx()
for f in filters:
    tax.fill_between(f.lam.to("angstrom").value, f.t, alpha=0.15)
    ax.plot(f.pivwv().to("angstrom").value, galaxy.photo_fnu["total"][f.filter_code].to("nJy").value,
            "ko", ms=5)
tax.set_ylim(0, 1.6)
tax.set_ylabel("filter transmission")
plt.show()

In [ ]:
# raw and psf are synthesizer's sums of the stars' and AGN images, whose filter order comes from
# a Python set and so changes between sessions: show every stage in FILTER_CODES order
for title, images in (("no PSF, no noise", raw), ("PSF", psf), ("PSF + noise", noisy)):
    images = ImageCollection(resolution=images.resolution, fov=images.fov,
                             imgs={fc: images[fc] for fc in FILTER_CODES})
    fig, ax = images.plot_images(show=False, ncols=7, individual_norm=True)
    fig.suptitle(title)
plt.show()

## 9. Saving the spectrum and photometry

As in the walkthrough.

In [ ]:
from smock.io import load_spectrum, save_spectrum

SIM = "ASTRID"
# the settings, the same for every galaxy of a run (a batch can write them once), then this galaxy's values
settings = {
    "simulation": SIM, "snapshot": SNAP, "redshift": z, "particles": PARTICLES, "centre": gal.centre_method,
    "star_mass": STAR_MASS, "dust_to_metal": DUST_TO_METAL, "sph_kernel": SPH_KERNEL,
    "kappa_code": KAPPA_CODE, "kappa_pc2_per_Msun": kappa, "kappa_BC": KAPPA_BC, "Z_BC": Z_BC,
    "young_age_Myr": YOUNG_AGE.to("Myr").value, "dust_slope_BC": DUST_SLOPE_BC, "dust_slope_ISM": DUST_SLOPE_ISM,
    "dust_temperature_K": DUST_TEMPERATURE.to("K").value, "dust_beta": DUST_BETA,
    "grid": GRID_NAME,
    **({"agn_grid": AGN_GRID_NAME, "agn_grid_point": {k: float(np.asarray(getattr(v, "value", v)))
                                                     for k, v in AGN_GRID_POINT.items()},
        "agn_u_from": AGN_U_FROM, "efficiency": EFFICIENCY, "disc_temperature": str(DISC_TEMPERATURE), "disc_radius": DISC_RADIUS, "disc_radius_unit": DISC_RADIUS_UNIT,
        "r_isco_rg": R_ISCO, "covering_fraction": COVERING_FRACTION, "covering_mode": COVERING_MODE,
        "covering_seed": COVERING_SEED, "agn_dust_slope": AGN_DUST_SLOPE,
        "agn_dust_window_A": None if AGN_DUST_WINDOW is None else [w.to("angstrom").value for w in AGN_DUST_WINDOW]}
       if INCLUDE_AGN else {}),
    "igm": IGM.__name__, "cosmology": {"H0": cosmo.H0.value, "Om0": cosmo.Om0},
}
galaxy_meta = {
    "sidx": SIDX, "M_star_subfind": m_star_subfind, "centre_ckpch": gal.centre_ckpch, "crop": gal.crop, "agn": agn_on,
    **({"bh_ids": [int(i) for i in ids], "los_covered": los_covered.tolist()} if agn_on else {}),
}
meta = {**settings, **galaxy_meta}
path = save_spectrum(OUT_DIR / f"{SIM}_{SNAP:03d}_sidx{SIDX}_spectrum.npz",
                     {c: owner[c].spectra[c] for c in SAVED},
                     {c: owner[c].photo_fnu[c] for c in SAVED}, meta)

spectrum, photometry, meta_back = load_spectrum(path)
print(path.name, "|", len(spectrum), "wavelengths |", len(photometry), "filters")
print("spectrum columns:", ", ".join(spectrum.dtype.names[:7]), "...")
print("photometry, total [AB]:", "  ".join(f"{f.split('.')[-1]} {m:.2f}" for f, m in
                                          zip(photometry["filter"], photometry["total_mag_AB"])))

## 10. Image segmentation

As in the walkthrough.

In [ ]:
# photutils imports tqdm, which warns in Jupyter when ipywidgets is missing; harmless
warnings.filterwarnings("ignore", message="IProgress not found")
from smock import analysis

DETECT_FILTER = "JWST/NIRCam.F444W"
DETECT_NSIGMA = 1.5             # the old pipeline's detection and deblending settings
DETECT_NPIXELS = 10
DETECT_CONNECTIVITY = 4         # 4: pixels touch by sides only; 8 counts corners too
DEBLEND_NLEVELS = 32
DEBLEND_CONTRAST = 0.001
IMAGE_CENTRE = "catalogue"      # or "light": centre the saved images on our source's centroid
DETECT_SMOOTH = None            # or "psf": find the sources on a copy smoothed with the filter's PSF (smoother
                                # segment edges); the threshold stays the unsmoothed one, so sources near the
                                # limit can be lost. Only the segmentation uses the copy
detect = noisy[DETECT_FILTER].arr                       # nJy per pixel
sigma_detect = (depths[DETECT_FILTER] / 5).to("nJy").value   # the per-pixel sigma from step 7

segm = analysis.segment(detect, sigma_detect, nsigma=DETECT_NSIGMA, npixels=DETECT_NPIXELS,
                        connectivity=DETECT_CONNECTIVITY, nlevels=DEBLEND_NLEVELS, contrast=DEBLEND_CONTRAST,
                        smooth_kernel=psfs[DETECT_FILTER] if DETECT_SMOOTH == "psf" else None)
segmentation_ok = segm is not None
if segmentation_ok:
    label, centroid, offset_px = analysis.pick_source(detect, segm)
    source = segm.data == label
else:
    label, centroid, offset_px = 0, analysis.image_centre(detect.shape), 0.0
    source = np.ones(detect.shape, dtype=bool)

kpc_per_arcsec = cosmo.kpc_proper_per_arcmin(z).to("kpc/arcsec").value   # physical kpc
pixel_kpc = PIXEL * kpc_per_arcsec
n_sources = segm.n_labels if segmentation_ok else 0
if segmentation_ok:
    print(f"{n_sources} sources; ours is label {label}, {source.sum()} pixels")
    print(f"its centroid is {offset_px:.2f} px = {offset_px * PIXEL:.3f}\" = {offset_px * pixel_kpc:.3f} pkpc "
          f"= {offset_px * pixel_kpc * (1 + z) * snap.h:.3f} ckpc/h from the catalogue position")
else:
    print("nothing detected: the measurements below use the whole image (segmentation_ok = False)")

# the FOV-wide window the saved images are cut to (step 13), with its first pixel (i0, j0)
n_out = round(FOV / PIXEL)
target = centroid if IMAGE_CENTRE == "light" else analysis.image_centre(detect.shape)
window = analysis.centred_window(n_out, detect.shape[0], target)
print(f"saved images: pixels {window[0]}-{window[0] + n_out - 1} (x), {window[1]}-{window[1] + n_out - 1} (y); "
      f"centroid at {centroid - window} in them, middle {(n_out - 1) / 2}")

from matplotlib.patches import Rectangle

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
extent = [-IMAGE_FOV / 2, IMAGE_FOV / 2, -IMAGE_FOV / 2, IMAGE_FOV / 2]
axes[0].imshow(detect.T, origin="lower", extent=extent, cmap="gray", vmin=-2 * sigma_detect, vmax=20 * sigma_detect)
corner = (window - 0.5 - analysis.image_centre(detect.shape)) * PIXEL
axes[0].add_patch(Rectangle(corner, FOV, FOV, fill=False, color="r", lw=1))
axes[0].set_title("F444W, PSF + noise")
if segmentation_ok:
    axes[1].imshow(segm.data.T, origin="lower", extent=extent, cmap=segm.cmap, interpolation="nearest")
axes[1].set_title(f"segmentation ({n_sources} sources)")
axes[2].imshow(np.where(source, detect, np.nan).T, origin="lower", extent=extent, cmap="viridis")
axes[2].plot(*((centroid - analysis.image_centre(detect.shape)) * PIXEL), "r+", ms=12)
axes[2].set_title("our source and its centroid")
for ax in axes:
    ax.set_xlabel("x [arcsec]")
axes[0].set_ylabel("y [arcsec]")
plt.show()

## 11. Galaxy Morphology (optional)

As in the walkthrough.

In [ ]:
RUN_STATMORPH = True

morph, morphology_ok = None, False
if RUN_STATMORPH and segmentation_ok:
    import statmorph
    from astropy.utils.exceptions import AstropyDeprecationWarning

    others = segm.data.astype(bool) & ~source
    # statmorph 0.7.2 calls a photutils function that photutils 3.1 deprecates (it still works)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", AstropyDeprecationWarning)
        morph = statmorph.SourceMorphology(detect, segm.data, label, mask=others,
                                           weightmap=np.full(detect.shape, sigma_detect),
                                           psf=psfs[DETECT_FILTER])
    morphology_ok = morph.flag <= 1
    print(f"flag {morph.flag}, Sersic flag {morph.flag_sersic}")
    print(f"r20, r50, r80 = {morph.r20 * PIXEL:.3f}, {morph.r50 * PIXEL:.3f}, {morph.r80 * PIXEL:.3f} arcsec "
          f"(circular, light)")
    print(f"C = {morph.concentration:.2f}, A = {morph.asymmetry:.3f}, S = {morph.smoothness:.3f}, "
          f"Gini = {morph.gini:.3f}, M20 = {morph.m20:.2f}")
    print(f"Sersic n = {morph.sersic_n:.2f}, R_e = {morph.sersic_rhalf * PIXEL:.3f} arcsec")

# for the files in step 13 (NaN, and flags -1, when statmorph didn't run); radii in arcsec
morphology = {f"morph_{k}": float(getattr(morph, k)) if morph is not None else np.nan
              for k in ("concentration", "asymmetry", "smoothness", "gini", "m20", "sersic_n")}
morphology |= {f"morph_{k}_arcsec": float(getattr(morph, k)) * PIXEL if morph is not None else np.nan
               for k in ("r20", "r50", "r80", "sersic_rhalf")}
morphology |= {f"morph_{k}": int(getattr(morph, k)) if morph is not None else -1 for k in ("flag", "flag_sersic")}


## 12. Re-measuring the galaxy's properties within the segmented area (pass 2)

As in the walkthrough, except the aperture spectrum's stellar part. With `APERTURE_SPECTRUM = "two_pass"` the chunks run through the model again, and only the stars inside the aperture are added up (chunks with none are skipped); with `"preset"` it was added up in pass 1, inside the preset circle. The black holes' part is a weighted sum, as in the walkthrough. `aperture_spectrum` in the saved files says which.

In [ ]:
from synthesizer.imaging import Image
from synthesizer.emissions import Sed
from unyt import kpc

APERTURE = "segment"            # the whole source; or "segment_and_r_half": its pixels within APERTURE_R_HALF r_half (the old pipeline)
APERTURE_R_HALF = 2.0           # the old pipeline: 2
SFR_AGE = 100 * Myr             # sfr_young: stars formed in the last SFR_AGE
COMPACTNESS_RADII = (0.1, 0.2)  # arcsec: compactness = f(< 0.2") / f(< 0.1"), as published
COMPACTNESS_METHOD = "exact"    # exact pixel overlaps; "taper" reproduces the old pipeline's aperture edges
IMAGE_PHOTOMETRY = ()           # also measure the photometry on the images, as an observer would: any of
                                # "psf" (noiseless, PSF-convolved) and "noisy", e.g. ("psf", "noisy")

# 1. maps on the images' pixel grid (the full IMAGE_FOV, pixels of pixel_kpc)
grid_kw = dict(resolution=pixel_kpc * kpc, fov=IMAGE_FOV * kpc_per_arcsec * kpc)
smooth_kw = dict(img_type="smoothed", kernel=kernel.get_kernel(), kernel_threshold=1)
maps = {
    "stellar_mass": galaxy.get_map_stellar_mass(**grid_kw, **smooth_kw).arr,        # Msun, current masses
    "gas_mass": galaxy.get_map_gas_mass(**grid_kw, **smooth_kw).arr,                # Msun
    "sfr_young": galaxy.get_map_sfr(**grid_kw, **smooth_kw, age_bin=SFR_AGE).arr,   # Msun/yr
}
sfr_map = Image(**grid_kw)                    # synthesizer has no gas-SFR map, so make one
sfr_map.generate_img_smoothed(signal=galaxy.gas.star_formation_rates.to("Msun/yr").value,
                              coordinates=galaxy.gas.centered_coordinates,
                              smoothing_lengths=galaxy.gas.smoothing_lengths,
                              kernel=kernel.get_kernel(), kernel_threshold=1)
maps["sfr_gas"] = sfr_map.arr                                                     # Msun/yr

# 2-3. r_half and the aperture
r_half_px = analysis.half_mass_radius(maps["stellar_mass"], source, centroid)
if APERTURE == "segment":
    aperture = source
else:
    aperture = source & (analysis.pixel_distances(source.shape, centroid) <= APERTURE_R_HALF * r_half_px)

# 4. sums inside the aperture
star_in = analysis.in_mask(galaxy.stars.centered_coordinates[:, :2].to("kpc").value, aperture, pixel_kpc)
bh = galaxy.black_holes                     # None when the galaxy has no BHs
bh_xy = bh.centered_coordinates[:, :2].to("kpc").value if bh is not None else np.zeros((0, 2))
bh_in = analysis.in_mask(bh_xy, aperture, pixel_kpc) if bh is not None else np.zeros(0, bool)
props = {
    "r_half_arcsec": r_half_px * PIXEL,
    "r_half_pkpc": r_half_px * pixel_kpc,
    "aperture_pixels": int(aperture.sum()),
    **{f"{name}_aperture": float(m[aperture].sum()) for name, m in maps.items()},
    "M_star_particles": float(galaxy.stars.current_masses[star_in].sum().to("Msun").value),
    "n_stars": int(star_in.sum()),
    "bh_ids": [int(i) for i in bh.ids[bh_in]] if bh is not None else [],
    "M_bh": float(bh.masses[bh_in].sum().to("Msun").value) if bh is not None else 0.0,
    "Mdot_bh": float(bh.accretion_rates[bh_in].sum().to("Msun/yr").value) if bh is not None else 0.0,
}

# the aperture spectrum. Stars: pass 2 runs the chunks again and adds up only the stars in
# the aperture (or, for "preset", pass 1 already did inside the preset circle). Black holes: a
# weighted sum, as in the walkthrough. Galaxy-level labels are sums of their parts (step 8).
if APERTURE_SPECTRUM == "two_pass":
    part_lnu = {p: 0.0 for p in STAR_LABELS}
    chunks = stars_only.split(chunk_stars)     # taken off one at a time, as in pass 1
    t0, start = time.time(), 0
    while chunks:
        chunk = chunks.pop(0)
        n_c = chunk.stars.nparticles
        weights = star_in[start:start + n_c].astype(float)
        start += n_c
        if not weights.any():
            continue
        with np.errstate(over="ignore"):
            chunk.get_spectra(stellar_dust)
        for p in STAR_LABELS:
            part_lnu[p] = part_lnu[p] + weights @ chunk.stars.particle_spectra[p].lnu.to("erg/s/Hz").value
        del chunk
    print(f"pass 2: {time.time() - t0:.0f} s")
    props["aperture_spectrum"] = "source aperture (two passes)"
else:
    part_lnu = dict(preset_lnu)
    props["aperture_spectrum"] = f"preset circle, r = {PRESET_APERTURE_ARCSEC} arcsec about the catalogue position"
for p in BH_LABELS:
    part_lnu[p] = bh_in.astype(float) @ galaxy.black_holes.particle_spectra[p].lnu.to("erg/s/Hz").value
aperture_seds, aperture_phot = {}, {}
for c in SAVED:
    lnu = sum(part_lnu[p] for p in parts[c])
    aperture_seds[c] = Sed(grid.lam, lnu=lnu * unyt_array(1.0, "erg/s/Hz"))
    aperture_seds[c].get_fnu(cosmo, z, igm=IGM)
    aperture_phot[c] = aperture_seds[c].get_photo_fnu(filters)

# 4b. photometry measured on the images: each filter's total image summed over the aperture [nJy]
stage_image = {"psf": psf, "noisy": noisy}
image_phot = {kind: {fc: float(stage_image[kind][fc].arr[aperture].sum()) for fc in FILTER_CODES}
              for kind in IMAGE_PHOTOMETRY}

# 5. compactness on the noiseless F444W image
bright_bh = None
if bh_in.any():
    k = np.argmax(np.where(bh_in, bh.accretion_rates.value, -np.inf))
    bright_bh = analysis.pixel_indices(bh_xy[k:k + 1], source.shape, pixel_kpc)[0]
centres = analysis.compactness_centres(centroid, bright_bh)
props["compactness"], best_centre, _ = analysis.compactness(psf[DETECT_FILTER].arr, source, centres, PIXEL,
                                                            COMPACTNESS_RADII, COMPACTNESS_METHOD)

for key, value in props.items():
    print(f"{key:22s} {value:.4g}" if isinstance(value, float) else f"{key:22s} {value}")
print(f"compactness centre: {best_centre} [px]")
for kind, phot in {"particles": {fc: aperture_phot["total"][fc].to("nJy").value for fc in FILTER_CODES},
                   **{f"{kind} image": phot for kind, phot in image_phot.items()}}.items():
    print(f"{kind:>12s} photometry in the aperture [nJy]: " + "  ".join(f"{fc.split('.')[-1]} {f:.2f}" for fc, f in phot.items()))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
units = {"stellar_mass": "Msun", "gas_mass": "Msun", "sfr_young": "Msun/yr", "sfr_gas": "Msun/yr"}
for ax, (name, m) in zip(axes, maps.items()):
    shown = np.where(m > 0, m, np.nan)
    ax.imshow(np.log10(shown).T, origin="lower", extent=extent, cmap="magma")
    ax.contour(source.T.astype(float), levels=[0.5], colors="w", linewidths=0.8, extent=extent)
    ax.contour(aperture.T.astype(float), levels=[0.5], colors="c", linewidths=0.8, extent=extent)
    ax.set_title(f"log {name} [{units[name]} / pixel]")
    ax.set_xlabel("x [arcsec]")
axes[0].set_ylabel("y [arcsec]")
fig.suptitle("property maps: source (white), aperture (cyan)")
plt.show()

## 13. Saving the spectrum and the image data

As in the walkthrough.

In [ ]:
from smock.io import image_stack, load_images, save_images

# settings, then this galaxy's values, as in step 9
source_settings = {
    "detect_filter": DETECT_FILTER, "detect_nsigma": DETECT_NSIGMA, "detect_npixels": DETECT_NPIXELS,
    "detect_connectivity": DETECT_CONNECTIVITY, "deblend_nlevels": DEBLEND_NLEVELS,
    "deblend_contrast": DEBLEND_CONTRAST, "aperture_r_half": APERTURE_R_HALF, "sfr_young_age_Myr": SFR_AGE.to("Myr").value,
    "compactness_radii_arcsec": COMPACTNESS_RADII, "compactness_method": COMPACTNESS_METHOD,
    "detect_smooth": DETECT_SMOOTH, "image_photometry": list(IMAGE_PHOTOMETRY),
}
source_values = {
    "segmentation_ok": segmentation_ok, "source_label": label,
    "aperture": APERTURE if segmentation_ok else "whole_image",
    "centroid_offset_xy_arcsec": (centroid - analysis.image_centre(detect.shape)) * PIXEL,
    "offset_px": offset_px, "morphology_ok": morphology_ok, **morphology, **props,
    "image_aperture_fnu_nJy": {kind: {fc.split(".")[-1]: f for fc, f in phot.items()} for kind, phot in image_phot.items()},
}
source_meta = {**source_settings, **source_values}
save_spectrum(OUT_DIR / f"{SIM}_{SNAP:03d}_sidx{SIDX}_aperture_spectrum.npz",
              aperture_seds, aperture_phot, {**meta, **source_meta})

image_settings = {
    "pixel_arcsec": PIXEL, "fov_arcsec": FOV, "units": "nJy per pixel",
    "image_centre": IMAGE_CENTRE, "buffer_pix": BUFFER_PIX,
    "axes": "filter, x, y", "psf_source": PSF_SOURCE, "psf_product": PSF_PRODUCT, "noise_preset": NOISE_PRESET,
    "kernel_margin_pix": KERNEL_MARGIN_PIX,
    "sigma_pix_nJy": {fc.split(".")[-1]: (depths[fc] / 5).to("nJy").value for fc in FILTER_CODES},
}
image_values = {
    "pixel_pkpc": pixel_kpc, "window_start_ij_px": window,
    "centroid_ij_px": centroid - window,      # in the saved images
    "noise_seed": [SEED, SNAP, SIDX],
}
images_meta = {**meta, **source_meta, **image_settings, **image_values}
stage_images = {"total_raw": raw, "total_psf": psf, "total_noisy": noisy}
for label, prefix in (("stellar_total", "stellar"), ("AGN_total", "AGN")):
    if label in alone:
        stage_images[f"{prefix}_raw"], stage_images[f"{prefix}_psf"] = alone[label]
stages = {name: analysis.cut_window(image_stack(images, FILTER_CODES), window, n_out)
          for name, images in stage_images.items()}
# all background (0) when nothing was detected, so every file has one
segmap = analysis.cut_window(segm.data, window, n_out) if segmentation_ok else np.zeros((n_out, n_out), int)
path = save_images(OUT_DIR / f"{SIM}_{SNAP:03d}_sidx{SIDX}_images.npz", FILTER_CODES, stages, segmap, images_meta)

images, meta_back = load_images(path)
print(path.name, "|", {k: v.shape for k, v in images.items()})
print("F444W total in the noiseless PSF image:", f"{images['total_psf'][-1].sum():.2f} nJy;",
      "r_half from the file:", f"{meta_back['r_half_arcsec']:.3f}\"")

## 14. Creating Figures

As in the walkthrough.

In [ ]:
from matplotlib.colors import AsinhNorm, LogNorm, PowerNorm  # noqa: F401 (alternatives for RGB_NORM)
from smock import plotting


def RGB_NORM(channels):
    """ambra_lrd's stretch: asinh, vmin 0, vmax the brightest displayed pixel, linear width 3% of vmax."""
    vmax = channels.max()
    return AsinhNorm(linear_width=0.03 * vmax, vmin=0, vmax=vmax, clip=True)
# or a fixed one, e.g. RGB_NORM = LogNorm(vmin=0.05, vmax=50, clip=True)   (nJy per pixel)


RGB = None                      # (R, G, B) filter names; None: F444W/F200W/F115W at z <= 6.5, else F444W/F277W/F150W
STAGE = "total_noisy"           # or "total_psf", "total_raw", "stellar_psf", "AGN_psf", ...
DISPLAY_ARCSEC = 2.0            # width shown, out of the saved FOV
DISPLAY_CENTRE = "centroid"     # "image", "centroid", or (i, j) in pixels
SCALE_BAR = "kpc"               # None, "arcsec", "kpc" (physical) or "ckpc/h"
LABEL = f"z = {z:.1f}"          # free text, or None
SPECTRUM_LEGEND = plotting.SPECTRUM_LEGEND   # outside the axes, right; or e.g. dict(loc="lower left", ncol=2); None: none
BAND_CMAP = "rainbow"           # filter bands coloured short (purple) to long (red) wavelength

images, image_meta = load_images(OUT_DIR / f"{SIM}_{SNAP:03d}_sidx{SIDX}_images.npz")
spectrum, _, spectrum_meta = load_spectrum(OUT_DIR / f"{SIM}_{SNAP:03d}_sidx{SIDX}_spectrum.npz")
rgb_options = dict(stage=STAGE, rgb=RGB, norm=RGB_NORM, display_arcsec=DISPLAY_ARCSEC,
                   centre=DISPLAY_CENTRE, scale_bar=SCALE_BAR)

# one panel, with the compactness apertures (r = 0.1" cyan, 0.2" green) around the centre that
# compactness chose
ax = plotting.plot_rgb(images, image_meta, label=LABEL, **rgb_options)
for radius, colour in zip(COMPACTNESS_RADII, ("cyan", "lime")):
    plotting.add_circle(ax, best_centre - window, radius / PIXEL, color=colour)
plt.show()

# the RGB and its three channels
plotting.plot_2x2(images, image_meta, **rgb_options)
plt.show()

# the image beside the spectrum, with the filters' bands shaded
plotting.plot_image_and_spectrum(images, image_meta, spectrum, spectrum_meta,
                                 rgb_kwargs=dict(label=LABEL, **rgb_options),
                                 spectrum_kwargs=dict(filters=filters, legend=SPECTRUM_LEGEND, band_cmap=BAND_CMAP))
plt.show()

## 15. Is it a little red dot? (optional)

As in the walkthrough.

A note on subhalo 54437, the "LRD-like" galaxy in step 1's test set: even though it has properties that favour it appearing as an LRD (it is compact, with a bright BH), with the defaults it doesn't come out as one. This is likely due in part to modelling its AGN's emission with the default grid, standard broad-line-region clouds like those around AGN at low redshift, rather than the gas-enshrouded model (`gas_enshrouded_core`, step 1).

In [ ]:
from smock.io import ab_mag

LRD_SELECTION = "ambra"         # LaChance et al. 2026 (AMBRA); or "kokorev24" (Kokorev et al. 2024)
LRD_MAX_OFFSET_CKPCH = 10.0     # source checks: catalogue position to centroid, comoving kpc/h
LRD_PHOTOMETRY = "particles"    # the particles inside the aperture (noiseless, no PSF); or "psf" or "noisy",
                                # measured on the images (step 12's IMAGE_PHOTOMETRY must include it)

fluxes = ({fc: aperture_phot["total"][fc].to("nJy").value for fc in FILTER_CODES} if LRD_PHOTOMETRY == "particles"
          else image_phot[LRD_PHOTOMETRY])          # nJy
mags = {fc.split(".")[-1]: float(ab_mag(f)) for fc, f in fluxes.items()}
select = {"ambra": analysis.lrd_selection_ambra, "kokorev24": analysis.lrd_selection_kokorev24}[LRD_SELECTION]
colours = select(mags, props["compactness"])
checks = analysis.lrd_source_quality(segmentation_ok, props["M_bh"], props["stellar_mass_aperture"], m_star_subfind,
                                     offset_px * pixel_kpc * (1 + z) * snap.h, max_offset_ckpch=LRD_MAX_OFFSET_CKPCH)

print("  ".join(f"{band} {m:.2f}" for band, m in mags.items()), f"| compactness {props['compactness']:.2f}")
print("selection:    ", {k: bool(v) for k, v in colours.items()})
print("source checks:", {k: bool(v) for k, v in checks.items()})
print(f"LRD ({LRD_SELECTION}, {LRD_PHOTOMETRY} photometry, with the source checks): {bool(colours['lrd'] and checks['ok'])}")